# ADIB anode-free substrate screening — Phase 1

**Setup.** A true ADIB has an Al metal negative electrode, AlCl₄⁻ intercalating into a graphite cathode, and a
chloroaluminate electrolyte. The plated species must be aluminium (Al₂Cl₇⁻ + e⁻ → Al), but the **substrate** it plates
onto, the anodic current collector, can be changed. This notebook screens those substrate materials.

**Phase 1** benchmarks Ridge, Random Forest and XGBoost on formation energy per atom, a property with full Materials
Project coverage. The goal is to prove the pipeline and pick a model before Phase 2's sparse, project-specific targets.

What this notebook does:

1. **Leakage audit** runs before any model is trained. `energy_above_hull` is excluded, because for elements it equals
   the target exactly. If any feature is identical to the target, the run stops.
2. **5-fold CV**, both shuffled KFold (random) and GroupKFold by chemical system (grouped), reported as mean ± std.
   The gap between the two shows whether the model generalises to unseen chemistries.
3. **Magpie** composition features (132), with two feature sets compared side by side: **full** (including DFT
   and structure features) and **composition-only**, which is what Phase 2 needs.

The code is copied verbatim from `substrate_screening.py`; the command-line script still works on its own.

## 0. Settings

- `DEMO = True` runs on **synthetic** data. The numbers only prove the pipeline works; they say nothing about real materials.
- `DEMO = False` pulls real entries from Materials Project. It needs `MP_API_KEY`, either set here or as an environment variable.
- `COMPOSITION_ONLY = True` runs only Magpie + `n_elements` and skips the side-by-side comparison.

In [10]:
import os

DEMO = True
MP_API_KEY = os.environ.get("MP_API_KEY")
COMPOSITION_ONLY = False

OUT = "model_comparison.csv"
AUDIT_OUT = "leakage_audit.csv"

## 1. Imports and configuration

In [11]:
import warnings
import sys
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

SEED = 42

N_FOLDS = 5

TOP_IMPORTANCES = 15

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

print("XGBoost available:", HAS_XGB)

XGBoost available: True


**Literature ground truth.** These are the Phase 3 validation set. If the model ranks them correctly, its
predictions on untested materials become credible.

In [12]:
KNOWN_SUBSTRATES = [
    ("W",              "stable",   "Established oxidatively stable collector in ADIBs"),
    ("Mo",             "stable",   "Established oxidatively stable collector in ADIBs"),
    ("glassy carbon",  "stable",   "Established oxidatively stable collector in ADIBs"),
    ("TiN",            "stable",   "Oxidative stability on par with W and Mo in AlCl3:EMIMCl"),
    ("graphite paper", "works",    "Cycles stably as anodic current collector, anode-free cell"),
    ("Ni",             "mixed",    "Cycles, but performance differs by corrosion resistance"),
    ("Cu",             "mixed",    "Cycles, but performance differs by corrosion resistance"),
    ("Ag",             "mixed",    "Cycles, but performance differs by corrosion resistance"),
    ("Mg",             "mixed",    "Cycles, but performance differs by corrosion resistance"),
    ("stainless steel","corrodes", "Corrodes in chloroaluminate melts"),
    ("Al",             "corrodes", "Oxidised in Al anolytes at high cell voltages"),
    ("Fe",             "corrodes", "Oxidised in Al anolytes at high cell voltages"),
]

**Candidate chemistries.** Add systems to this list to widen the search.

In [33]:
CANDIDATE_SYSTEMS = [
    "Ti-N", "Zr-N", "Hf-N", "V-N", "Nb-N", "Ta-N", "Cr-N",     # nitrides
    "Ti-C", "W-C", "Mo-C", "V-C", "Nb-C", "Ta-C", "Si-C",      # carbides
    "Ti-B", "Zr-B", "Hf-B", "Cr-B",                            # borides
    "W", "Mo", "Ni", "Cu", "Ag", "Ti", "Ta", "Nb", "Cr",       # elemental metals
]

**Feature groups and the leakage exclusion.**

- `STRUCTURE_DFT_FEATURES` need a relaxed structure, and all except `nsites` need a DFT calculation. They are dropped in composition-only mode.
- `LEAKY_EXCLUDED` columns are downloaded but must never be model inputs. Every feature set is checked against this list.

The hand-written `mean_Z`, `max_Z` and `min_Z` are gone, because Magpie's mean, minimum and maximum atomic number are the same quantities.

In [14]:
STRUCTURE_DFT_FEATURES = ["density", "vol_per_atom", "band_gap", "nsites"]

COMPOSITION_EXTRA = ["n_elements"]

LEAKY_EXCLUDED = {
    "energy_above_hull": (
        "E_hull = E_form - E_hull_convex(composition); computed from this "
        "entry's formation energy and its competitors'. Elemental hulls sit "
        "at 0, so E_hull == E_form exactly for every elemental entry."
    ),
}

EXACT_MATCH_FRAC = 0.01

SINGLE_FEATURE_R2 = 0.80

GROUP_CONST_FRAC = 0.95

## 2. Data

**Synthetic generator (demo).**

- Each group is a real chemical system with its own offset added to the target, so an unseen family really is
  harder to predict than a familiar one. On this data the random-to-grouped gap must come out positive.
- Elemental systems are referenced to their ground state, and `energy_above_hull` is computed from the target.
  So, as in Materials Project, `energy_above_hull` equals the target on every elemental row, and the audit has a real leak to catch.

In [15]:
_Z = {"B": 5, "C": 6, "N": 7, "Si": 14, "Ti": 22, "V": 23, "Cr": 24, "Ni": 28,
      "Cu": 29, "Zr": 40, "Nb": 41, "Mo": 42, "Ag": 47, "Hf": 72, "Ta": 73,
      "W": 74}

_METALS = ["Ti", "V", "Cr", "Ni", "Cu", "Zr", "Nb", "Mo", "Ag", "Hf", "Ta", "W"]

_ANIONS = ["B", "C", "N", "Si"]

_RATIOS = [(1, 1), (2, 1), (1, 2), (3, 1), (3, 4), (5, 3), (4, 3)]

def make_synthetic(n=900, n_groups=25):
    """SYNTHETIC DATA - pipeline smoke test only, not real materials.

    Ground truth is deliberately nonlinear so the escalation ladder is visible:
    Ridge should underperform the tree models.

    Each group is a chemical system with its own offset added to the target,
    the way real families share a bias. A model can learn that offset from
    family-identifying features when the family is in training, but not for an
    unseen family - so the grouped split must score LOWER than the random one.
    If the random-to-grouped gap is not clearly positive here, the gap
    diagnostic is broken.

    energy_above_hull is generated FROM the target (distance above the lowest
    target in its group), mirroring how the real quantity is derived, and
    elemental systems are referenced to their ground state. Together that makes
    E_hull == target on every elemental row, exactly as in Materials Project,
    so the leakage audit has something real to catch.

    Only density, vol_per_atom and band_gap carry signal beyond the family
    offset, so composition-only scores here are poor by construction.
    """
    rng = np.random.default_rng(SEED)

    systems = [(m,) for m in _METALS] + [(m, a) for m in _METALS for a in _ANIONS]
    chosen = [systems[i] for i in rng.choice(len(systems), n_groups, replace=False)]
    group_offset = rng.normal(0, 0.4, n_groups)

    gidx = rng.integers(0, n_groups, n)
    formulas, mean_z, n_el = [], [], []
    for g in gidx:
        sys_ = chosen[g]
        if len(sys_) == 1:
            amounts = {sys_[0]: 1}
        else:
            p, q = _RATIOS[rng.integers(len(_RATIOS))]
            amounts = {sys_[0]: p, sys_[1]: q}
        total = sum(amounts.values())
        formulas.append("".join(f"{el}{k if k > 1 else ''}" for el, k in amounts.items()))
        mean_z.append(sum(_Z[el] * k for el, k in amounts.items()) / total)
        n_el.append(len(amounts))

    df = pd.DataFrame({
        "formula":      formulas,
        "group":        ["-".join(sorted(chosen[g])) for g in gidx],
        "n_elements":   n_el,
        "density":      rng.uniform(2, 18, n),
        "vol_per_atom": rng.uniform(8, 30, n),
        "nsites":       rng.integers(1, 40, n),
        "band_gap":     rng.gamma(1.2, 0.8, n),
    })
    mean_z = np.asarray(mean_z)

    y = (
        -0.35 * np.sqrt(df["density"])
        + 0.02 * df["vol_per_atom"]
        - 0.6 * np.tanh(df["band_gap"])
        - 0.004 * mean_z
        + 0.15 * np.sin(df["density"] / 2)
        + group_offset[gidx]
        + rng.normal(0, 0.12, n)
    )
    # Elements are referenced to their own ground state, so an elemental
    # system's lowest entry sits at exactly 0 - as in Materials Project.
    elemental = df["n_elements"] == 1
    y = y - np.where(elemental, y.groupby(df["group"]).transform("min"), 0.0)
    df["target"] = y
    df["energy_above_hull"] = df["target"] - df.groupby("group")["target"].transform("min")
    return df

**Materials Project fetch.** Rows with no formation energy are skipped; the target is never filled in with estimates.

In [16]:
def fetch_mp(api_key):
    """Pull real candidate substrates from Materials Project."""
    try:
        from mp_api.client import MPRester
    except ImportError:
        sys.exit("mp-api not installed. Run: pip install mp-api pymatgen")

    fields = ["material_id", "formula_pretty", "chemsys", "nsites", "density",
              "volume", "band_gap", "formation_energy_per_atom",
              "energy_above_hull", "elements", "theoretical"]

    rows = []
    with MPRester(api_key) as mpr:
        for chemsys in CANDIDATE_SYSTEMS:
            try:
                docs = mpr.materials.summary.search(chemsys=chemsys, fields=fields)
            except Exception as exc:
                print(f"  ! {chemsys:<8} failed: {exc}", file=sys.stderr)
                continue
            print(f"  {chemsys:<8} {len(docs):>5} entries")
            for d in docs:
                if d.formation_energy_per_atom is None:
                    continue                       # never impute the target
                rows.append({
                    "material_id": str(d.material_id),
                    "formula": d.formula_pretty,
                    "group": d.chemsys,            # group = chemical system
                    "density": d.density,
                    "vol_per_atom": d.volume / d.nsites if d.nsites else None,
                    "nsites": d.nsites,
                    "band_gap": d.band_gap,
                    "energy_above_hull": d.energy_above_hull,   # audit/filter only
                    "n_elements": len(d.elements),
                    "theoretical": d.theoretical,
                    "target": d.formation_energy_per_atom,
                })

    if not rows:
        sys.exit("No results. Check the API key and network.")

    df = pd.DataFrame(rows)
    n_elem_rows = int((df["n_elements"] == 1).sum())
    print(f"\n  {len(df)} entries, {n_elem_rows} elemental "
          f"({100 * n_elem_rows / len(df):.1f}% of rows)")
    return df

In [17]:
if DEMO:
    print("SYNTHETIC DEMO MODE - numbers describe a made-up function, not real materials.")
    df = make_synthetic()
else:
    if not MP_API_KEY:
        raise ValueError("Set MP_API_KEY (or DEMO = True).")
    print("MATERIALS PROJECT MODE - target: formation energy per atom")
    df = fetch_mp(MP_API_KEY)

print(f"{len(df)} rows, {df['group'].nunique()} chemical systems")
df.head()

SYNTHETIC DEMO MODE - numbers describe a made-up function, not real materials.
900 rows, 25 chemical systems


,formula,group,n_elements,density,vol_per_atom,nsites,band_gap,target,energy_above_hull
0,Ta4Si3,Si-Ta,2,16.053852,29.391617,16,0.994769,-1.022120,1.005486
1,TaSi,Si-Ta,2,8.064843,27.863639,23,1.122766,-1.377203,0.650403
2,Ta4Si3,Si-Ta,2,10.272312,28.281745,21,0.624347,-1.197695,0.829911
3,Ta5Si3,Si-Ta,2,13.867463,15.307587,5,0.641082,-1.257169,0.770437
4,Ta3N,N-Ta,2,13.698176,11.720964,30,0.608410,-1.490990,0.209964


In [18]:
df.groupby("group").agg(rows=("target", "size"),
                        n_elements=("n_elements", "first"),
                        target_mean=("target", "mean")).round(3).T

group,Ag,Ag-C,Ag-N,Ag-Si,B-Cu,B-Mo,B-Ni,C-Cr,C-Cu,C-Hf,...,N-Ni,N-Ta,N-W,Nb,Nb-Si,Ni,Si-Ta,Si-W,Si-Zr,Ta
rows,49.000,35.000,32.000,43.000,29.000,25.000,33.000,26.000,35.000,38.00,...,37.000,34.000,36.000,24.000,37.000,37.000,46.000,49.000,45.000,34.000
n_elements,1.000,2.000,2.000,2.000,2.000,2.000,2.000,2.000,2.000,2.00,...,2.000,2.000,2.000,1.000,2.000,1.000,2.000,2.000,2.000,1.000
target_mean,0.769,-1.579,-0.824,-1.463,-1.332,-0.281,-1.284,-0.892,-1.232,-0.84,...,-0.928,-1.066,-1.094,0.561,-1.068,0.599,-1.166,-1.107,-1.268,0.592


## 3. Magpie features and feature sets

- **full**: Magpie + `n_elements` + `density`, `vol_per_atom`, `band_gap`, `nsites`
- **composition-only**: Magpie + `n_elements`

Density, band gap and volume come from the same DFT relaxation that produces formation energy. For a genuinely new
material you can't get them without running DFT, and that run gives you formation energy anyway. So the full set
is circular for predicting new materials, and composition-only is the set that matters for Phase 2.

In [19]:
def add_magpie(df):
    """Append the 132 Magpie ElementProperty features. Returns (df, labels)."""
    try:
        from matminer.featurizers.composition import ElementProperty
        from pymatgen.core import Composition
    except ImportError:
        sys.exit("matminer not installed. Run: pip install matminer pymatgen")

    ep = ElementProperty.from_preset("magpie", impute_nan=True)
    labels = ep.feature_labels()
    uniq = df["formula"].unique()                  # polymorphs share a formula
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        table = pd.DataFrame([ep.featurize(Composition(f)) for f in uniq],
                             index=uniq, columns=labels)
    df = df.join(table, on="formula")
    print(f"  Magpie: {len(labels)} features from {len(uniq)} unique formulas")
    return df, labels

def feature_sets(df, magpie_labels, composition_only):
    """Build the feature sets, dropping columns that are constant in this data."""
    constant = [c for c in magpie_labels if df[c].nunique(dropna=True) <= 1]
    magpie = [c for c in magpie_labels if c not in constant]
    if constant:
        print(f"  dropped {len(constant)} Magpie features constant across "
              f"this dataset")
    sets = {"composition-only": magpie + COMPOSITION_EXTRA}
    if not composition_only:
        sets = {"full": magpie + COMPOSITION_EXTRA + STRUCTURE_DFT_FEATURES, **sets}
    return sets

In [20]:
df, magpie_labels = add_magpie(df)
sets = feature_sets(df, magpie_labels, COMPOSITION_ONLY)

for name, cols in sets.items():
    overlap = set(cols) & set(LEAKY_EXCLUDED)
    if overlap:
        raise RuntimeError(f"Leaky columns in feature set '{name}': {sorted(overlap)}")

all_features = list(dict.fromkeys(c for cols in sets.values() for c in cols))
before = len(df)
df = df.dropna(subset=all_features + ["target"]).reset_index(drop=True)
if len(df) < before:
    print(f"  dropped {before - len(df)} rows with missing features")

y, groups = df["target"], df["group"]
print(f"{len(df)} samples, {groups.nunique()} groups, {N_FOLDS}-fold CV")
for name, cols in sets.items():
    print(f"  {name:<17} {len(cols)} features")

  Magpie: 132 features from 144 unique formulas
  dropped 9 Magpie features constant across this dataset
900 samples, 25 groups, 5-fold CV
  full              128 features
  composition-only  124 features


## 4. Leakage audit

Each column is checked against the target before any model sees it.

| flag | meaning | action |
|---|---|---|
| `LEAK:identity` | the column equals the target on more than 1% of rows (rows where both are 0 are ignored) | **halts** |
| `review:solo-R2` | the column alone reaches grouped R² > 0.8 | warning only; elements vs compounds separate legitimately |
| `group-proxy` | the column is constant within a chemical system | not leakage, but it lets the random split memorise families |

In [21]:
def audit_leakage(df, columns, groups):
    """Screen each column for target leakage before any model sees it.

    Three checks per column:
      exact     share of rows where the column equals the target (1e-6 tol),
                ignoring rows where both are 0 - elemental ground states have
                E_form = 0, and so do many range/avg_dev Magpie stats and every
                metal's band gap, which is coincidence rather than leakage
      solo_R2   grouped-split R2 of a small forest trained on that column alone
      grp_const share of multi-member groups in which the column never varies

    exact above threshold -> target leakage; the run halts.
    solo_R2 above threshold -> warning only. A strong single predictor is not
    proof of leakage: elements sit at E_form >= 0 while nitrides and borides
    are strongly negative, so n_elements alone reaches R2 ~0.8 legitimately.
    Review these by hand for a derivation from the target.
    grp_const above threshold -> the column is a chemical-system label. That is
    not target leakage, but it lets the RANDOM split memorise families, so read
    the grouped numbers.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    rows = []
    for col in columns:
        sub = pd.DataFrame({"x": df[col], "y": df["target"], "g": groups}).dropna()
        both_zero = np.isclose(sub["x"], 0) & np.isclose(sub["y"], 0)
        exact = float(np.mean(np.isclose(sub["x"], sub["y"], atol=1e-6) & ~both_zero))
        rho = sub["x"].corr(sub["y"], method="spearman")

        tr, te = next(gss.split(sub, sub["y"], sub["g"]))
        m = RandomForestRegressor(n_estimators=50, min_samples_leaf=5,
                                  random_state=SEED, n_jobs=-1)
        m.fit(sub[["x"]].iloc[tr], sub["y"].iloc[tr])
        solo_r2 = r2_score(sub["y"].iloc[te], m.predict(sub[["x"]].iloc[te]))

        sizes = sub.groupby("g")["x"].size()
        multi = sizes[sizes > 1].index
        if len(multi):
            grp_const = float((sub[sub["g"].isin(multi)]
                               .groupby("g")["x"].nunique() == 1).mean())
        else:
            grp_const = float("nan")

        flags = []
        if exact > EXACT_MATCH_FRAC:
            flags.append("LEAK:identity")
        if solo_r2 > SINGLE_FEATURE_R2:
            flags.append("review:solo-R2")
        if grp_const >= GROUP_CONST_FRAC:
            flags.append("group-proxy")
        rows.append({
            "column": col,
            "role": "excluded" if col in LEAKY_EXCLUDED else "feature",
            "exact": exact, "spearman": rho, "solo_R2": solo_r2,
            "grp_const": grp_const, "flags": ",".join(flags) or "ok",
        })
    return pd.DataFrame(rows)

In [22]:
audit = audit_leakage(df, all_features + list(LEAKY_EXCLUDED), groups)
audit.to_csv(AUDIT_OUT, index=False)
print(f"{int((audit['flags'] == 'ok').sum())} of {len(audit)} columns clean; full table in {AUDIT_OUT}")
audit["flags"].str.split(",").explode().value_counts()

60 of 129 columns clean; full table in leakage_audit.csv


flags
group-proxy       61
ok                60
review:solo-R2    22
LEAK:identity      1
Name: count, dtype: int64

In [23]:
shown = audit[audit["flags"].str.contains("LEAK|review") | (audit["role"] == "excluded")]
shown.assign(column=shown["column"].str.replace("MagpieData ", "", regex=False)).round(3)

,column,role,exact,spearman,solo_R2,grp_const,flags
0,minimum Number,feature,0.000,0.464,0.803,1.0,"review:solo-R2,group-proxy"
7,maximum MendeleevNumber,feature,0.000,-0.499,0.802,1.0,"review:solo-R2,group-proxy"
12,minimum AtomicWeight,feature,0.000,0.464,0.803,1.0,"review:solo-R2,group-proxy"
14,range AtomicWeight,feature,0.000,-0.465,0.811,1.0,"review:solo-R2,group-proxy"
25,maximum Column,feature,0.000,-0.507,0.807,1.0,"review:solo-R2,group-proxy"
32,range Row,feature,0.000,-0.438,0.807,1.0,"review:solo-R2,group-proxy"
34,avg_dev Row,feature,0.000,-0.424,0.806,0.2,review:solo-R2
36,minimum CovalentRadius,feature,0.000,0.430,0.802,1.0,"review:solo-R2,group-proxy"
54,maximum NpValence,feature,0.000,-0.510,0.812,1.0,"review:solo-R2,group-proxy"
55,range NpValence,feature,0.000,-0.510,0.812,1.0,"review:solo-R2,group-proxy"


In [24]:
for col, why in LEAKY_EXCLUDED.items():
    print(f"excluded {col}: {why}\n")

bad = audit[(audit["role"] == "feature") & audit["flags"].str.contains("LEAK")]
if not bad.empty:
    raise RuntimeError(f"Features flagged for target leakage: {list(bad['column'])}. "
                       "Investigate before modelling.")

review = audit[(audit["role"] == "feature") & audit["flags"].str.contains("review")]
if not review.empty:
    print(f"{len(review)} features predict the target strongly on their own. Not halting - "
          "no identity with the target - but check none is derived from it.")

proxies = audit[(audit["role"] == "feature") & audit["flags"].str.contains("group-proxy")]
if not proxies.empty:
    print(f"{len(proxies)} of {len(all_features)} features are group proxies (constant within a "
          "chemical system).\nThe random split can memorise families through them - trust the grouped CV.")
print("\nNo feature is identical to the target - safe to model.")

excluded energy_above_hull: E_hull = E_form - E_hull_convex(composition); computed from this entry's formation energy and its competitors'. Elemental hulls sit at 0, so E_hull == E_form exactly for every elemental entry.

22 features predict the target strongly on their own. Not halting - no identity with the target - but check none is derived from it.
61 of 128 features are group proxies (constant within a chemical system).
The random split can memorise families through them - trust the grouped CV.

No feature is identical to the target - safe to model.


## 5. Models and cross-validation

- Every model is scored with 5-fold shuffled KFold (**random**) and 5-fold GroupKFold by chemical system (**grouped**).
- In grouped folds, Ridge's alpha is picked by a grouped CV on the training fold only. With leave-one-out tuning,
  Ridge had memorised families, chosen a tiny alpha, and scored R² ≈ −1000 on one held-out fold.

In [25]:
def build_models(ridge_cv=None):
    """ridge_cv: splits used to pick Ridge's alpha (None = efficient LOO).

    Under grouped evaluation this must be a grouped split of the training fold.
    LOO lets group-proxy features memorise families, picks a tiny alpha, and
    the model then extrapolates wildly on held-out chemistries (observed:
    R2 of -1000 on a fold, predictions of ~28 eV/atom).
    """
    models = {
        "Ridge": make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=np.logspace(-3, 5, 33), cv=ridge_cv),
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2,
            random_state=SEED, n_jobs=-1,
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBRegressor(
            n_estimators=600, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            random_state=SEED, n_jobs=-1, tree_method="hist",
        )
    else:
        models["GradientBoosting*"] = GradientBoostingRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=4,
            subsample=0.8, random_state=SEED,
        )
    return models

METRICS = {
    "R2":   r2_score,
    "MAE":  mean_absolute_error,
    "RMSE": lambda a, b: float(np.sqrt(mean_squared_error(a, b))),
}

def evaluate(X, y, groups):
    """5-fold CV of every model, shuffled KFold and GroupKFold by chemsys.

    In grouped folds Ridge's alpha is chosen by an inner GroupKFold on the
    training fold only, so tuning never sees the held-out chemistries.

    Returns one row per model with <split>_<metric> (fold mean) and
    <split>_<metric>_std (fold std).
    """
    outer = {
        "random":  list(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(X)),
        "grouped": list(GroupKFold(N_FOLDS).split(X, y, groups)),
    }
    scores = {}                                    # (model, split, metric) -> list
    for split, folds in outer.items():
        for tr, te in folds:
            ridge_cv = None
            if split == "grouped":
                ridge_cv = list(GroupKFold(N_FOLDS).split(
                    X.iloc[tr], y.iloc[tr], groups.iloc[tr]))
            for name, model in build_models(ridge_cv).items():
                model.fit(X.iloc[tr], y.iloc[tr])
                pred = model.predict(X.iloc[te])
                for metric, fn in METRICS.items():
                    scores.setdefault((name, split, metric), []).append(
                        fn(y.iloc[te], pred))

    rows = []
    for name in build_models():
        rec = {"model": name}
        for split in outer:
            for metric in METRICS:
                vals = np.asarray(scores[(name, split, metric)])
                rec[f"{split}_{metric}"] = vals.mean()
                rec[f"{split}_{metric}_std"] = vals.std()
        rows.append(rec)
    return pd.DataFrame(rows)

def pm(row, col, digits=3):
    return f"{row[col]:.{digits}f} +/- {row[col + '_std']:.{digits}f}"

def results_table(results):
    out = pd.DataFrame({"model": results["model"]})
    for split in ("random", "grouped"):
        for metric in METRICS:
            out[f"{split}_{metric}"] = [pm(r, f"{split}_{metric}")
                                        for _, r in results.iterrows()]
    return out

Running the models takes a minute or two on the demo data, and longer on Materials Project data.

In [26]:
results = {name: evaluate(df[cols], y, groups) for name, cols in sets.items()}

combined = pd.concat([r.assign(feature_set=n) for n, r in results.items()], ignore_index=True)
combined.to_csv(OUT, index=False)

for name, r in results.items():
    print(f"Model comparison - {name} ({len(sets[name])} features, mean +/- std over {N_FOLDS} folds)")
    display(results_table(r))

Model comparison - full (128 features, mean +/- std over 5 folds)


,model,random_R2,random_MAE,random_RMSE,grouped_R2,grouped_MAE,grouped_RMSE
0,Ridge,0.922 +/- 0.010,0.182 +/- 0.006,0.231 +/- 0.013,0.397 +/- 0.396,0.389 +/- 0.150,0.468 +/- 0.168
1,RandomForest,0.953 +/- 0.005,0.143 +/- 0.006,0.180 +/- 0.006,0.618 +/- 0.221,0.291 +/- 0.077,0.382 +/- 0.122
2,XGBoost,0.962 +/- 0.005,0.128 +/- 0.006,0.162 +/- 0.007,0.492 +/- 0.248,0.340 +/- 0.106,0.465 +/- 0.180


Model comparison - composition-only (124 features, mean +/- std over 5 folds)


,model,random_R2,random_MAE,random_RMSE,grouped_R2,grouped_MAE,grouped_RMSE
0,Ridge,0.766 +/- 0.026,0.324 +/- 0.014,0.401 +/- 0.013,0.131 +/- 0.548,0.471 +/- 0.135,0.564 +/- 0.148
1,RandomForest,0.742 +/- 0.028,0.337 +/- 0.009,0.422 +/- 0.013,0.272 +/- 0.461,0.417 +/- 0.054,0.516 +/- 0.069
2,XGBoost,0.722 +/- 0.029,0.349 +/- 0.009,0.437 +/- 0.014,0.151 +/- 0.456,0.468 +/- 0.076,0.586 +/- 0.115


## 6. Feature importance (top 15)

The remaining features are summed into one row. Treat the Ridge column as a rough signal only: its coefficients are
spread unpredictably across about 125 overlapping Magpie features.

In [27]:
def importances(X, y, top=TOP_IMPORTANCES):
    """Normalised importances from each model fitted on all rows.

    Top `top` features by mean importance across models; the rest are summed
    into one row. Ridge coefficients over ~130 collinear Magpie features are
    unstable - read its column as a rough signal, not an attribution.
    """
    out = {}
    for name, m in build_models().items():
        m = clone(m).fit(X, y)
        if hasattr(m, "feature_importances_"):
            imp = m.feature_importances_
        else:                                      # Ridge pipeline
            coef = np.abs(m.steps[-1][1].coef_)
            imp = coef / coef.sum()
        out[name] = imp
    tab = pd.DataFrame(out, index=X.columns.str.replace("MagpieData ", "", regex=False))
    tab = tab.loc[tab.mean(axis=1).sort_values(ascending=False).index]
    head = tab.head(top).copy()
    if len(tab) > top:
        head.loc[f"(other {len(tab) - top} features)"] = tab.iloc[top:].sum()
    return head.round(4)

In [28]:
for name, cols in sets.items():
    print(f"Feature importance - {name} (top {TOP_IMPORTANCES}, normalised)")
    display(importances(df[cols], y))

Feature importance - full (top 15, normalised)


,Ridge,RandomForest,XGBoost
minimum Number,0.0137,0.0337,0.2464
range NUnfilled,0.0037,0.0224,0.1348
range Number,0.0009,0.0162,0.1282
density,0.0019,0.1225,0.0110
range Column,0.0040,0.0110,0.0650
avg_dev Row,0.0378,0.0234,0.0015
minimum AtomicWeight,0.0187,0.0426,0.0003
maximum NdValence,0.0001,0.0002,0.0588
maximum NpValence,0.0415,0.0159,0.0000
minimum CovalentRadius,0.0106,0.0443,0.0006


Feature importance - composition-only (top 15, normalised)


,Ridge,RandomForest,XGBoost
minimum Number,0.0179,0.0461,0.1511
range NUnfilled,0.0525,0.0305,0.1253
avg_dev NUnfilled,0.0451,0.0348,0.0158
maximum SpaceGroupNumber,0.0395,0.0010,0.0494
minimum NdValence,0.0183,0.0354,0.0195
range Number,0.0061,0.0170,0.0483
avg_dev GSvolume_pa,0.0154,0.0289,0.0237
maximum MendeleevNumber,0.0072,0.0417,0.0172
minimum Row,0.0107,0.0504,0.0000
avg_dev MeltingT,0.0121,0.0319,0.0102


## 7. Cost of dropping DFT/structure features

On synthetic data the target mostly depends on the DFT-style features, so composition-only scores poorly
**by construction**. Only real data shows the actual cost.

In [29]:
if len(results) == 2:
    full, comp = results["full"], results["composition-only"]
    side = pd.DataFrame({
        "model": full["model"],
        "full grouped_R2": [pm(r, "grouped_R2") for _, r in full.iterrows()],
        "comp-only grouped_R2": [pm(r, "grouped_R2") for _, r in comp.iterrows()],
        "delta_R2": (comp["grouped_R2"] - full["grouped_R2"]).round(3),
        "full grouped_MAE": [pm(r, "grouped_MAE") for _, r in full.iterrows()],
        "comp-only grouped_MAE": [pm(r, "grouped_MAE") for _, r in comp.iterrows()],
    })
    display(side)
else:
    print("COMPOSITION_ONLY = True - no side-by-side comparison.")

,model,full grouped_R2,comp-only grouped_R2,delta_R2,full grouped_MAE,comp-only grouped_MAE
0,Ridge,0.397 +/- 0.396,0.131 +/- 0.548,-0.266,0.389 +/- 0.150,0.471 +/- 0.135
1,RandomForest,0.618 +/- 0.221,0.272 +/- 0.461,-0.346,0.291 +/- 0.077,0.417 +/- 0.054
2,XGBoost,0.492 +/- 0.248,0.151 +/- 0.456,-0.341,0.340 +/- 0.106,0.468 +/- 0.076


## 8. Interpretation

- Headline the grouped number.
- If the gap is within the fold-to-fold spread, it can't be told apart from zero.
- With about 25 chemical systems, each grouped fold holds out only about 5, so grouped fold std is large. For real
  data, consider repeated GroupKFold or leave-one-group-out.

In [30]:
def interpret(results, label):
    best = results.loc[results["grouped_R2"].idxmax()]
    ridge = results[results["model"] == "Ridge"].iloc[0]
    gap = best["random_R2"] - best["grouped_R2"]
    noise = max(best["random_R2_std"], best["grouped_R2_std"])

    print(f"\n[{label}] Best on grouped CV: {best['model']} "
          f"(R2 {pm(best, 'grouped_R2')}, MAE {pm(best, 'grouped_MAE')})")
    print(f"  Random-to-grouped R2 gap: {gap:.3f} (largest fold std {noise:.3f})")
    if abs(gap) <= noise:
        print("  -> Gap is within fold-to-fold spread; it cannot be distinguished "
              "from zero.")
    elif gap > 0.15:
        print("  -> Large gap. The model leans on chemical-family similarity.")
        print("     Report the GROUPED number as your headline result.")
    elif gap > 0:
        print("  -> Modest gap beyond fold noise. Some reliance on family "
              "similarity; headline the grouped number.")
    else:
        print("  -> Grouped scores above random beyond fold noise. Unexpected - "
              "check the group labels.")

    if best["model"] != "Ridge" and best["grouped_R2"] - ridge["grouped_R2"] < 0.05:
        print(f"  Ridge is within 0.05 R2 of {best['model']} "
              f"(Ridge {pm(ridge, 'grouped_R2')}). Prefer Ridge - the")
        print("  interpretability is worth more than the marginal accuracy, and")
        print("  it is a defensible choice to justify in a viva.")

In [31]:
for name, r in results.items():
    interpret(r, name)

print(f"\nWritten to {OUT} and {AUDIT_OUT}")


[full] Best on grouped CV: RandomForest (R2 0.618 +/- 0.221, MAE 0.291 +/- 0.077)
  Random-to-grouped R2 gap: 0.335 (largest fold std 0.221)
  -> Large gap. The model leans on chemical-family similarity.
     Report the GROUPED number as your headline result.

[composition-only] Best on grouped CV: RandomForest (R2 0.272 +/- 0.461, MAE 0.417 +/- 0.054)
  Random-to-grouped R2 gap: 0.470 (largest fold std 0.461)
  -> Large gap. The model leans on chemical-family similarity.
     Report the GROUPED number as your headline result.

Written to model_comparison.csv and leakage_audit.csv


## 9. Phase 3 validation set (literature ground truth)

In [32]:
pd.DataFrame(KNOWN_SUBSTRATES, columns=["substrate", "verdict", "note"])

,substrate,verdict,note
0,W,stable,Established oxidatively stable collector in ADIBs
1,Mo,stable,Established oxidatively stable collector in ADIBs
2,glassy carbon,stable,Established oxidatively stable collector in ADIBs
3,TiN,stable,Oxidative stability on par with W and Mo in Al...
4,graphite paper,works,"Cycles stably as anodic current collector, ano..."
5,Ni,mixed,"Cycles, but performance differs by corrosion r..."
6,Cu,mixed,"Cycles, but performance differs by corrosion r..."
7,Ag,mixed,"Cycles, but performance differs by corrosion r..."
8,Mg,mixed,"Cycles, but performance differs by corrosion r..."
9,stainless steel,corrodes,Corrodes in chloroaluminate melts
